
### Learning delta time travel and disaster recovery

在传统的关系型数据库或普通大数据文件夹里，“数据被覆盖（Overwrite）”或者“误删（Delete）”是一场物理意义上的单向毁灭。一旦 commit 成功，老数据在磁盘上的磁道就会被直接抹去并填入新字节。此时想找回历史状态，只能寄希望于极其笨重、需要停机几个小时的数据库全量备份物理恢复。

而 Delta Lake 凭借“不变量肉身（Immutable Parquet）+ 追加式账本（Delta Log）”的设计，强行在分布式存储中拉出了一个【多维时空】。
在你执行 UPDATE 或 OVERWRITE 时，老文件根本没有被擦除，它依然完好无损地躺在云端硬盘里！大管家只是在最新的 JSON 账本里把它标记为“对当前时间线不可见”。
这就意味着，只要历史物理文件没有被 VACUUM 强行清空，你就能随时拿着历史版本号当做“时空通行证”，命令引擎直接穿透当前错乱的时间线，一秒钟降维打击、无痛复活任何历史瞬间！

In [0]:
print("模拟灾难发生前夜")

spark.table("gold_seller_dimension").show()

In [0]:
print("故意增加一个误操作：实习生将所有total_sales改为0元")

spark.sql("UPDATE gold_seller_dimension SET total_sales == 0.0")

In [0]:
print("故意增加一个误操作：误删全表")

# 注：这个delete和delete table是有区别的，delete是删除表的数据，delete from是删除整张表，包括表结构
spark.sql("DELETE FROM gold_seller_dimension")
print("模拟灾难发生后")

spark.table("gold_seller_dimension").show()

调用 Delta Lake 的时空账本，揪出灾难发生前的那个黄金版本号


In [0]:
df_history = spark.sql("DESCRIBE HISTORY gold_seller_dimension")
df_history.select("version", "timestamp", "userId", "operation", "operationMetrics").show(truncate=False)

|6      |2026-06-16 09:24:37|71725956393057|DELETE                           |{numRemovedFiles -> 1, numRemovedBytes -> 1686, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, executionTimeMs -> 38, numDeletionVectorsUpdated -> 0, numAddedFiles -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, numDeletedRows -> 4, scanTimeMs -> 28, numAddedBytes -> 0, rewriteTimeMs -> 0}                                                                                                                                                                                                      


                                                                                                                                                      |
|5      |2026-06-16 09:23:29|71725956393057|UPDATE                           |{numRemovedFiles -> 1, numRemovedBytes -> 1525, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, executionTimeMs -> 3572, numDeletionVectorsUpdated -> 0, scanTimeMs -> 54, numAddedFiles -> 1, numUpdatedRows -> 4, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, numAddedBytes -> 1686, rewriteTimeMs -> 3465}   

可以看到一条是version：5 update，一条是version 6 delete ，因此我们需要回溯至version4

In [0]:
print("准备回溯至version 4")

df_v4 = (
    spark.read
    .option("versionAsOf", 4)
    .table("gold_seller_dimension")  # 完整名如sales.gold.gold_seller_dimension
)

df_v4.show()

可以看到回溯成功，数据暂时存入df_v4, 然后后续直接写入，变成version 7


In [0]:
df_v4.write.format("delta").mode("overwrite").saveAsTable("gold_seller_dimension")

print("Success!")

In [0]:
display(spark.table("gold_seller_dimension"))